# BONUS: Troubleshooting reference, Spark UI, run history, failure patterns

This reference demo accompanies the hands-on <a href="$../lab/lab_troubleshooting">lab_troubleshooting</a>. It covers exam domain 6 (Troubleshooting, Monitoring & Optimization).

| Section | Topic |
|---------|-------|
| 1 | Query profile (serverless) and Spark UI (classic): stages, shuffle read/write, spill |
| 2 | Jobs run-history trends (done in class, together with section 3) |
| 3 | Failure patterns: symptom, cause, fix, Spark UI vs Query Profile (done in class) |
| 4 | System tables pointers |

## 1. Reading the Spark UI

On classic compute, open the cluster and click **Spark UI** (or the *View* link on a cell). On serverless, open the cell's query profile. The query profile shows operator-level metrics: rows, time, shuffle, spill and join type. It has no Stages tab and no per-task spread, so you cannot compare max and median task time there.

> **In this course we use serverless.** Steps 2 and 3 below (Stages tab, task summary metrics) exist only on classic compute. You see them on screenshots and learn them for exam recognition.

Read the UI in this order:

1. Open the SQL / DataFrame tab and the slow query, then read the plan graph bottom-up. Check rows output per node (explosions show up here), scan pruning and join strategy (Broadcast vs SortMerge/ShuffleHash).
2. Open the Stages tab (classic only) and sort by duration. A job is a chain of stages split at shuffle boundaries: `groupBy`, non-broadcast `join`, `distinct`, `repartition` and window functions.
3. Inside a stage (classic only), open Summary metrics for tasks and compare max with median.

| Stage metric | Healthy | Trouble sign | Means |
|---|---|---|---|
| Task time max vs median | about equal | max ≫ median | skew: one partition is much larger (hot key) |
| Shuffle read/write | proportional to data | inflated vs input | wide operations move too much (bad key, no broadcast, explosion) |
| Spill (memory/disk) | 0 | > 0 | partitions do not fit in task memory, data is written to disk mid-stage |
| Task count | about data size / 128 MB | thousands of tasks of a few ms | over-partitioned shuffle (see lab Task 4) |

Usual first fixes:

- Skew: AQE skew join (on by default), or salt the key.
- Big shuffle: broadcast the small side and prune columns early.
- Spill: use more partitions or more memory per task. Fewer, bigger partitions make it worse.
- Explosion: fix the join key, not the cluster.

The table in the next cell is not the point: 1000 keys with 2000 rows each. Below the cell, click **See performance** → the statement → **See query profile**, and find the **Shuffle** node between the two **Grouping Aggregate** nodes.

In [0]:
# Generate a small shuffle so there is something to inspect in the UI/profile.
from pyspark.sql import functions as F

df = spark.range(0, 2_000_000).withColumn("key", (F.col("id") % 1000))
agg = df.groupBy("key").agg(F.count("*").alias("cnt"), F.avg("id").alias("avg_id"))
display(agg.orderBy(F.desc("cnt")).limit(5))

# Serverless: open this cell's query profile (See performance -> query). Look for Aggregate and
# Shuffle/Exchange (the shuffle boundary) with rows, time, shuffle bytes and spill per operator.
# There is no per-task max vs median spread there: that is Spark UI -> Stages -> Summary metrics
# for tasks, on classic compute only.

> **Note:** Serverless exposes only a short list of Spark confs, so most of them show `not readable` in the table below. On serverless, AQE is managed and always on (coalesce partitions, skew join, dynamic join selection), and shuffle partitions are `auto`.
> On classic compute the cell prints the real values. AQE is on by default since DBR 7.3.

In [0]:
# Check that the AQE flags are on, because they fix many of these problems for you.
rows = []
for k in [
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.autoBroadcastJoinThreshold",
    "spark.sql.shuffle.partitions",
]:
    try:
        rows.append((k, spark.conf.get(k, "(not set)")))
    except Exception:   # serverless exposes only a few Spark confs (CONFIG_NOT_AVAILABLE)
        rows.append((k, "not readable (managed by Databricks on serverless, AQE is always on)"))
display(spark.createDataFrame(rows, "conf STRING, value STRING"))

## 2. Jobs run-history trends

> **In class (Day 3, before <a href="$../lab/lab_troubleshooting">lab_troubleshooting</a>):** sections 2 and 3. They answer three questions in order. Did it get slower? (run history) Why did it fail or slow down? (table in section 3) Where do I see it? (Spark UI on classic compute, Query Profile on serverless.) Exam objectives 6.1, 6.2, 6.3 and 6.5.

One failed run is an incident. A trend is a diagnosis. Open **Jobs & Pipelines**, click the job name, then **Runs**. The UI steps (matrix view, hover a task, Graph view, **Repair run**) are in `08_job_orchestration`, section *Run History*.

| What you see in the matrix view | Likely cause | First check |
|---|---|---|
| Duration creep: each run is a bit slower than the last | Input grows without layout maintenance (small files, no `OPTIMIZE` / clustering), or streaming state grows | Task duration vs its earlier runs; file count of the input table (`DESCRIBE DETAIL`) |
| Step change right after a deploy | The deployed change, not the data | Diff of that deploy |
| Failures at the same hour | Collision with another workload (quota, concurrent writes to one table) | Other jobs scheduled at that time |
| Red task, the tasks after it show Upstream failed | One root failure, the others never ran | Error first line of the red task (lab Task 1) |

Compare a task with its own earlier runs (the baseline), not with other tasks. A run can succeed and still be the problem when its duration doubled. After a fix, **Repair run** reruns only the failed tasks and their dependents.

## 3. Failure patterns: symptom → cause → fix

Use this triage rule first. If task time is 0 seconds, the cause is infrastructure: read the compute event log. If the error comes before any stage ran, the cause is analysis or permissions: read the error's first line. If stages ran but were slow, the cause is performance or data: read the stage metrics and the query profile.

<img src="../../../assets/images/nbts-triage.png" alt="A run failed or got slow: task time 0 s points to the compute event log, an error before any stage to the first line of the error, slow stages to the Query Profile and four common causes" width="800">

| Failure | Symptom | Cause | Where you see it | Fix |
|---|---|---|---|---|
| **Data skew** | One task (or a few) runs much longer than the rest; the stage waits for it | A hot key: one value holds a large share of the rows (`NULL`, `'unknown'`, a big customer) | Classic: **Stages** → stage → **Summary metrics for tasks**, max ≫ median duration and shuffle read. Serverless: query profile shows the time in the aggregate / join, but no per-task spread | AQE skew join (on by default); filter or handle the hot key separately; salt the key |
| **Large shuffle** | Long stage with large **Shuffle Read / Write** | Wide operations (`groupBy`, non-broadcast `join`, `distinct`, window) move much more data than needed | Classic: Stages tab, shuffle columns. Serverless: query profile, shuffle / exchange operator, bytes written | Broadcast the small side; select and filter columns before the join; aggregate before the join |
| **Row-exploding join** (lab Task 3) | Join output ≫ input rows, shuffle write inflated, write after the join slow | Join key not unique on one side (truncated or wrong key) | Classic: **SQL / DataFrame** tab, rows out of the join node. Serverless: query profile, join node rows | Join on the real key; check key cardinality with a count |
| **Disk spill** | **Spill (Memory) / Spill (Disk)** > 0, stage slower than expected | A partition does not fit in the task memory, rows are written to disk mid-stage | Classic: Stages tab, spill columns. Serverless: query profile, spill metric of the operator | More (smaller) shuffle partitions, fix skew, more memory per task |
| **Too many shuffle partitions** (lab Task 4) | Thousands of tiny tasks, each a few ms | `spark.sql.shuffle.partitions` set far above data size | Classic: task count of the stage. Serverless: query profile, partition count | `auto` (AQE coalesces partitions) or a value sized to the data |
| **OOM, executor** *(classic)* | Task retries, then `java.lang.OutOfMemoryError` | Huge partitions, a big table broadcast, heavy spill before it | Stage metrics (spill, input per task), executor logs | Fix skew; do not broadcast a big table (`autoBroadcastJoinThreshold`); more partitions; bigger workers last |
| **OOM, driver** *(classic)* | Notebook dies or cluster restarts after `collect()` / `toPandas()` | All rows pulled to the driver | Driver logs; the cell that collects | Write to a table; `limit()` for samples; bigger driver last |
| **Cluster start failure** *(classic)* | Job fails with 0 s task time: *Cluster terminated: init script failure / capacity* | Init script exits non-zero, no instance capacity, bad config | **Compute → Event log** (+ init script log). Not the notebook, not the Spark UI | Fix or remove the init script; other node type or zone; instance pool. Serverless avoids this class |
| **Library conflict** | `ModuleNotFoundError`, `AttributeError` after someone installed a library; pip errors at start | Two versions of one library on a shared cluster | **Compute → Event log** / library status; serverless: environment install errors in the run output | Pin exact versions; notebook-scoped `%pip`; serverless: one **environment** per notebook or job |
| **UC permission denied** | `PERMISSION_DENIED: User does not have SELECT on table …` | A privilege missing in the chain | The error names the securable; Catalog Explorer → **Permissions** | `GRANT USE CATALOG` + `USE SCHEMA` + `SELECT` |
| **Path / table not found** (lab Tasks 1, 5) | `PATH_NOT_FOUND`, `UC_VOLUME_NOT_FOUND`, `TABLE_OR_VIEW_NOT_FOUND` | Wrong name or path, missing `depends_on`, or no access (UC hides the object) | Error first line; `SHOW TABLES`, Volume listing | Fix the name, use 3-part names, add the dependency |
| **VACUUM vs time travel** | `VERSION AS OF` fails (`DELTA_UNSUPPORTED_TIME_TRAVEL_BEYOND_DELETED_FILE_RETENTION_DURATION`, `FileNotFoundException`) | The version is older than `delta.deletedFileRetentionDuration` (VACUUM may already have deleted its files) | `DESCRIBE HISTORY` | Keep `delta.deletedFileRetentionDuration` (default 7 days) ≥ how far back you time-travel |

On classic compute, the Spark UI **Stages** tab gives one row per stage. Inside a stage, **Summary metrics for tasks** show min, median and max of duration, shuffle read and spill. Skew means max ≫ median.

On serverless there is no Spark UI. Click **See performance** under the cell output, then the query, then **Query profile**. It shows operators (scan, join, aggregate, exchange) with rows, time, shuffle bytes and spill, but not the per-task spread. The exam question about stage-level metrics refers to the classic Spark UI.

In [0]:
# Build a skewed key (90% of rows are key 0), because a window has no partial step before the
# shuffle.
skewed_df = spark.range(1_000_000).selectExpr("id", "IF(id % 10 = 0, id + 1, 0) AS key")
skewed_df.createOrReplaceTempView("skewed")
q = "SELECT key, row_number() OVER (PARTITION BY key ORDER BY id) AS rn FROM skewed"
plan = spark.sql(f"EXPLAIN {q}").first()[0]
print(plan)
top = spark.sql(f"SELECT key, MAX(rn) AS n FROM ({q}) GROUP BY key ORDER BY n DESC LIMIT 1")
top = top.first()
print(f"hot key {top.key}: {top.n:,} of 1,000,000 rows go through one window partition")

Read the plan bottom-up. `Exchange hashpartitioning(key, …)` is the shuffle, then `Sort` and `Window` run per key. On serverless, Photon runs these steps, so the plan shows them as `PhotonShuffleExchangeSink hashpartitioning(key, …)`, `PhotonSort` and `PhotonWindow`. The query profile calls them Shuffle, Sort and Window. A window function has no partial step, so all 900,000 rows of key 0 cross the shuffle and land in one task. Each of the other keys holds a single row.

That task is the max in the stage's task summary on classic compute. On serverless, click **See performance** under the cell. It lists every statement of the cell (`createOrReplaceTempView`, the `EXPLAIN` and `top.first()`). Only the `top = top.first()` row runs the window, so open that one, then **Query profile**, and look at the Window and Shuffle operators.

A `GROUP BY key` with `COUNT(*)` on the same data barely shows skew. Its plan has a partial aggregate (`partial_count`) before the `Exchange`, so each map task sends one pre-counted row per key. Skew hurts where whole rows of a hot key must meet in one place: windows, sort-merge joins on the key, `collect_list`.

> **Exam shape:** a skewed `groupBy` that still shows one huge task with AQE on → salt the key (module 06). AQE skew handling covers joins only. Partial aggregation shrinks the shuffle for count/sum, not for windows, `collect_list` or `count(DISTINCT)`.

In [0]:
# Trigger an error that is safe anywhere, then read its error class first.
from pyspark.errors import PySparkException

# 3a. The error class names the problem. It is the text in brackets at the start of the first
# line, the same line you see in job run output and logs.
try:
    spark.table("system.information_schema.no_such_table").count()
except PySparkException as e:
    msg = str(e)
    print(msg[max(msg.find("["), 0):].splitlines()[0])  # skip the Java exception name serverless may add
else:
    print("Not rejected: system.information_schema.no_such_table was found")

**3b. Permission problems do not always say `PERMISSION_DENIED`.** Participants have no grants on the `system` catalog. A participant who runs the query below gets an error such as `INSUFFICIENT_PERMISSIONS` (*User does not have USE CATALOG on Catalog 'system'* or *USE SCHEMA on Schema 'system.compute'*).

Unity Catalog can also report an object you cannot see, or a system schema that is not enabled or not populated yet, as `TABLE_OR_VIEW_NOT_FOUND` or `SCHEMA_NOT_FOUND`. Check the names first, then the grant chain: `USE CATALOG` + `USE SCHEMA` + `SELECT`.

> **Demo (trainer only):** run the query below with an account that has `USE CATALOG` on `system`, `USE SCHEMA` on `system.compute` and `SELECT`.

```python
spark.sql("SELECT * FROM system.compute.clusters LIMIT 1").collect()
```

## 4. System tables pointers

System tables (`system` catalog, UC-governed, account-wide) turn "click through the UI" into SQL. You can also build alerts and dashboards on them.

| Table | What it answers |
|---|---|
| `system.lakeflow.jobs` / `pipelines` | Inventory: every job and pipeline, creator, tags |
| `system.lakeflow.job_run_timeline` | Run outcomes and durations over time (the trend table) |
| `system.lakeflow.job_task_run_timeline` | The same per task, so you see which task degrades |
| `system.billing.usage` (+ `list_prices`) | DBU per workload, for cost regressions after a "fix" |
| `system.access.audit` | Who ran, changed or granted what, and when |
| `event_log(...)` (TVF) | Per-pipeline events: expectations pass/drop rates, backlog |

Two traps when you query `job_run_timeline`. The tables are account-wide, so filter `workspace_id`. A run longer than 1 hour is split into hourly rows, and `result_state` is set only on the final row, so aggregate per `run_id` before you count runs. Job names live in `system.lakeflow.jobs` (`run_name` is NULL for runs of saved jobs).

> **Demo (trainer only):** the query cell below. System tables hold the job history of every workspace in the account region, including other teams' jobs and their `run_as`, so the course does not grant them to the training group.

An admin must grant `USE CATALOG` on `system`, `USE SCHEMA` on the schema (for example `system.lakeflow`) and `SELECT`. The system schema must also be enabled. On a new workspace the tables fill with a delay, and until then you get `TABLE_OR_VIEW_NOT_FOUND` even with correct grants.

Pipeline events are in the Beta system table `system.lakeflow_pipeline_events_preview.pipeline_events`. For one pipeline, use the `event_log(...)` TVF. `system.lakeflow.pipeline_update_timeline` has pipeline update outcomes.

In [0]:
# Demo (trainer only): failure rate and duration trend per job for the last 7 days.
# Needs USE CATALOG on system, USE SCHEMA on system.lakeflow and SELECT.
try:
    from databricks.sdk import WorkspaceClient
    # System tables are account-wide, so filter on workspace_id (STRING column).
    ws_id = WorkspaceClient().get_workspace_id()
    display(spark.sql(f"""
        WITH runs AS (            -- one row per run: hourly rows collapsed, final state via MAX_BY
          SELECT workspace_id, job_id, run_id,
                 MIN(period_start_time) AS start_time,
                 MAX(period_end_time)   AS end_time,
                 MAX_BY(result_state, period_end_time) AS result_state
          FROM system.lakeflow.job_run_timeline
          WHERE workspace_id = '{ws_id}' AND period_start_time >= current_date() - INTERVAL 7 DAYS
          GROUP BY ALL
        ),
        jobs AS (                 -- latest name per job
          SELECT workspace_id, job_id, name,
                 ROW_NUMBER() OVER (PARTITION BY workspace_id, job_id
                                    ORDER BY change_time DESC) AS rn
          FROM system.lakeflow.jobs
          WHERE workspace_id = '{ws_id}'
        )
        SELECT COALESCE(j.name, '(one-time run)') AS job_name,
               DATE(r.start_time) AS run_date,
               COUNT(*) AS runs,
               COUNT_IF(r.result_state IN ('FAILED','ERROR','TIMED_OUT')) AS failed,
               ROUND(100 * COUNT_IF(r.result_state IN ('FAILED','ERROR','TIMED_OUT')) / COUNT(*), 1)
                   AS failure_rate_pct,
               ROUND(AVG(TIMESTAMPDIFF(SECOND, r.start_time, r.end_time)) / 60, 1) AS avg_minutes
        FROM runs r
        LEFT JOIN jobs j
          ON r.workspace_id = j.workspace_id AND r.job_id = j.job_id AND j.rn = 1
        WHERE r.result_state IS NOT NULL      -- finished runs only
        GROUP BY ALL ORDER BY run_date DESC, runs DESC LIMIT 20
    """))
except Exception as e:
    msg = str(e)
    print(f"system.lakeflow query failed: {msg[max(msg.find('['), 0):].splitlines()[0][:150]}")
    if "PERMISSION_DENIED" in msg or "INSUFFICIENT_PERMISSIONS" in msg:
        print("Cause: missing grant -> ask an admin for USE CATALOG on system")
        print("       + USE SCHEMA on system.lakeflow + SELECT")
    elif "NOT_FOUND" in msg:
        print("Cause: the table is not visible here -> the system schema is not enabled yet,")
        print("       the tables are not populated yet (new workspace: can take a while),")
        print("       or you lack USE SCHEMA/SELECT (UC hides it).")
    # Fallback: recent runs from the Jobs API
    # (last 10 runs you can view; one final state per run, no trend)
    try:
        import itertools
        from databricks.sdk import WorkspaceClient
        runs = []
        for r in itertools.islice(WorkspaceClient().jobs.list_runs(expand_tasks=False, limit=10), 10):
            state = r.state.result_state.value if r.state and r.state.result_state else None
            runs.append((r.run_name, state, r.run_duration))
        print("Fallback: last 10 runs via the Jobs API")
        display(spark.createDataFrame(
            runs, "run_name STRING, result_state STRING, run_duration_ms BIGINT"))
    except Exception as e2:
        print(f"SDK fallback not available here: {e2}")

## Summary

| Question | First place to look |
|---|---|
| Why did it fail? | Run details, first red task, error first line |
| Why is it slow? | Serverless: query profile (operator rows/time, shuffle, spill, join type). Classic: Spark UI stage metrics for skew (max ≫ median), shuffle size and spill |
| Why did it get slower? | Run history matrix, or `system.lakeflow.job_run_timeline` trends |
| Why did it not even start? | Classic: Compute Event log (init scripts, capacity, libraries). Serverless: run output and environment errors |
| Who can or cannot read it? | Error's securable name, then Catalog Explorer permissions or `system.access.audit` |

> **Exam:** know which UI surface answers which question, the shuffle/spill/skew vocabulary and repair-run semantics. System tables are the queryable history of all of it.

← <a href="$./10_governance_and_security">10 — Governance & Security</a> | **<a href="$../../..">Course folder (README)</a>** | <a href="$../lab/lab_troubleshooting">lab_troubleshooting →</a>

## Self-check (trainer): participants can skip this cell

In [0]:
# Self-check (trainer): repeats the error demo and checks the values the demo cells saved

# Skew: the window plan has a shuffle, and key 0 holds 900,000 rows in one window partition
assert "Exchange" in plan and "Window" in plan, plan
assert top.key == 0 and top.n == 900_000, top

# Error class: the missing table is still reported as TABLE_OR_VIEW_NOT_FOUND
try:
    spark.table("system.information_schema.no_such_table").count()
except PySparkException as e:
    first = str(e).splitlines()[0]
    assert "[TABLE_OR_VIEW_NOT_FOUND]" in first, first
else:
    raise AssertionError("system.information_schema.no_such_table was found")

print("Self-check passed")